## Pre-attention mechanism being introduced with LLM's

**RNN's are example which are used to translate from one language to another**

- RNN's used encoder and decoder architecture, unlike LLM's which use decoder only architecture.
- The output from previous steps is fed as input for current step.
- The encoder processes the input text sequencially(one word at a time) and adjusts its hidden state(vector representation/context of the sentence) at each step.
- The output(context) of the encoder is the input for decoder.
- The decoder will build the sentence word by word. It will also adjust it's hidden state which stores the context from encoder and text generated till the current step.
- The decoder could not directly access the earlier hidden states of the encoder but only the final context it has produced. If the input text is long and complex, there will be information loss due to compression of context in each step.

- Later Bahdanau attention mechanism was introduced for RNN's.
- Where decoder part of the network can access all input tokens selectively, that means some inputs are given more importance that the others, which is determined by the attention weights(later).
- For example, for generating the 2nd output token there is more context in the 3rd input token, then it will be given more importance.

## Self Attention

- In self attention we are trying to find the relationship between one part of the input to the other parts of the input.

<img src="images/SelfAttentinOverview.png" width="500">

- In self-attention, our goal is to calculate context vectors z(i) for each element x(i)
in the input sequence. A context vector can be interpreted as an enriched embedding
vector.

- The embedding vector of the second input element, x(2) (which corresponds to the token “journey”), and the corresponding context vector, z(2), shown at the bottom of figure 3.7. This enhanced context vector, z(2), is an embedding that contains information about x(2) and all other input elements, x(1) to x(T).

- Their purpose is to create
enriched representations of each element in an input sequence (like a sentence)
by incorporating information from all other elements in the sequence.

<!-- - Each element in context vector will hold the information about its normal embedding vector and also how it is  -->

<img src="images/SequenceToContextVectors.png" width=600>

In [1]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

/Users/a.t.yaswanthreddy/VS Code/DreamTeam/LLM from scratch/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


<img src="images/AttentionScore.png" width=600>

- Figure 3.8 illustrates how we calculate the intermediate attention scores between the
query token and each input token. We determine these scores by computing the dot
product of the query, x(2), with every other input token.

- Dot product tells us how similar(direction) two vectors are.

In [2]:
query = inputs[1]
atten_score_2 = torch.empty(inputs.shape[0]) # If 10 inputs then we will have 10 attention scores
for i, x in enumerate(inputs):
    atten_score_2[i] = torch.dot(query, x)
print(atten_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [7]:
atten_weights_2_tmp = atten_score_2/atten_score_2.sum()
print("Attention weights" ,atten_weights_2_tmp)
print("Sum of weights ", atten_weights_2_tmp.sum())

Attention weights tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of weights  tensor(1.0000)


- The main goal behind the normalization is to obtain attention
weights that sum up to 1. 

- This normalization is a convention that is useful for interpretation and maintaining training stability in an LLM.

- After doing this we get attention weights.

- In practice it is more advisable to use softmax function to normalize. Which is better at handling extreme values.

In [10]:
atten_weights_2 = torch.softmax(atten_score_2, dim=0)
print("Attention weights: " ,atten_weights_2)
print("Sum: ", atten_weights_2.sum())

Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


- Context vector is the combined sum of all the products of embedded vectors of the token and the attention weight.

<img src="images/ContextVector.png" width=640>

In [18]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x in enumerate(inputs):
    context_vec_2 += atten_weights_2[i] * x
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Now we will calculate the attention weights for all input tokens

<img src="images/AttentionMatrix.png" width=600>

In [21]:
atten_scores = torch.zeros((6,6))
for i, x in enumerate(inputs):
    for j, y in enumerate(inputs):
        atten_scores[i][j] = torch.dot(x, y)
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Instead of using for loops as they are slow we use matrix multiplication.

In [23]:
atten_scores = inputs @ inputs.T
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [32]:
atten_weights = torch.softmax(atten_scores, dim=-1)
print(atten_weights)
print(atten_weights[0].sum())

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor(1.)


In [34]:
context_vec = atten_weights @ inputs
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## Computing the attention weights step by step

In [36]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

Now we initialize 3 weight matrices which will be optimized during training.

In [38]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [39]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


Weight parameters vs. attention weights
In the weight matrices W, the term “weight” is short for “weight parameters,” the val-
ues of a neural network that are optimized during training. This is not to be confused
with the attention weights. As we already saw, attention weights determine the extent
to which a context vector depends on the different parts of the input (i.e., to what
extent the network focuses on different parts of the input).
In summary, weight parameters are the fundamental, learned coefficients that define
the network’s connections, while attention weights are dynamic, context-specific values.

In [40]:
keys = inputs @ W_key
values = inputs @ W_value

print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


Here the query is of context vector 2

<br>
<img src="images/Attention-2.png" width=600>

In [42]:
atten_scores_2 = query_2 @ keys.T
print(atten_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [43]:
d_k = keys.shape[-1]
atten_weights_2 = torch.softmax(atten_scores_2 / d_k ** 0.5 , -1)
print(atten_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### The rationale behind scaled-dot product attention
The reason for the normalization by the embedding dimension size is to improve the
training performance by avoiding small gradients. For instance, when scaling up the
embedding dimension, which is typically greater than 1,000 for GPT-like LLMs, large
dot products can result in very small gradients during backpropagation due to the
softmax function applied to them. As dot products increase, the softmax function
behaves more like a step function, resulting in gradients nearing zero. These small
gradients can drastically slow down learning or cause training to stagnate.
The scaling by the square root of the embedding dimension is the reason why this
self-attention mechanism is also called scaled-dot product attention.

In [46]:
context_vec_2 = atten_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


- We create a query for each token.

- The query layer is used to know what information does this token need from the sentence. (For it as an example "it" token might be searching for a noun, not in a literal sense but as training goes it will reach a mathamatical conclusion which makes it see "it" as looking for somthing which is similar to a noun). This layer is used to formulate a question.

- In the key layer, if "it" is the query token then noun/subject token like "cat" might match more strongly. Keys can be seen as somthing that describes a token(Mathamatically explaining the "it" and noun relation).

- Value layer contains the actual information to retrive. Values are what that actually gets combined to produce the output representation.(If the model attends to “cat”, the value vector of “cat” contributes to the output).

- These layers are trained and during the training they develop these relationships.

```
Suppose we process the token “it”.
-- Step 1 — create query --
q_it

-- Step 2 — compare with all keys -- 
q_it ⋅ k_the
q_it ⋅ k_cat
q_it ⋅ k_sat

-- Step 3 — convert scores to weights -- 
cat → 0.8
sat → 0.1
the → 0.1

-- Step 4 — combine values -- 
output_it =
0.8 * v_cat
+0.1 * v_sat
+0.1 * v_the

So “it” now contains information from “cat.”
```